# 04 — Financial Model Integration
Trade finance default stacking ensemble, CCC/working-capital predictor, and the SC-PD credit risk scorer (Section D2.3, Deliverables D8-D10). Assumes the models from `03_model_training.ipynb` are already trained in-session, or re-trains them fresh below.

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
from src.data.synthetic_generator import SupplyChainDataGenerator
from src.models.gnn import train_gnn
from src.models.xgboost_model import train_xgboost_default_model
from src.models.survival import fit_cox_ph
from src.models.transformer import train_shipment_transformer

gen = SupplyChainDataGenerator(seed=42)
nodes, edges = gen.generate_graph()
lc_df = gen.generate_trade_finance_transactions(n=6000, nodes=nodes)
shipments = gen.generate_shipments(n=3000)

_, _, gnn_embeddings, gnn_node_ids, _ = train_gnn(nodes, edges, verbose=False)
xgb_model, _, _, _ = train_xgboost_default_model(nodes, n_trials=10, verbose=False)
cox_model, _, _ = fit_cox_ph(nodes)
transformer_model, _ = train_shipment_transformer(shipments, epochs=25, verbose=False)
print('Level-0 base learners ready')

Level-0 base learners ready


## Trade finance default prediction — stacking ensemble (Section A7.2, Deliverable D2.3.1)

In [2]:
from src.financial.trade_finance_default import train_trade_finance_default_model
meta_model, X, split, tf_metrics = train_trade_finance_default_model(
    lc_df, nodes, xgb_model, cox_model, gnn_embeddings, gnn_node_ids, transformer_model)
tf_metrics

{'oof_auc': 0.7505253891406438,
 'test_auc': 0.7484305872747934,
 'test_gini': 0.4968611745495868,
 'test_brier': 0.05894076903854951,
 'backend': 'lightgbm',
 'ece': 0.02267099570941618,
 'n_features': 24}

## Working capital / CCC predictor (Deliverable D2.3.2, Section A6.4 worked example)

In [3]:
from src.financial.ccc_predictor import train_ccc_predictor, predict_ccc_change, covenant_breach_alert
ccc_models, ccc_metrics = train_ccc_predictor()
print(ccc_metrics)

# MedDevice Corp scenario
signals = dict(delta_otif=-0.12, delta_lead_time_std=4.2, delta_freight_cost_ratio=0.0,
               delta_port_congestion=2.1, delta_inventory_turnover=0.0)
result = predict_ccc_change(ccc_models, signals, horizon_days=90)
alert = covenant_breach_alert(72, result['ccc_change'], 90)
print(result); print(alert)

  [CCC predictor] MAPE(30d)=39.24%  MAE=2.72 days
{'mape_30d': 39.24337128354401, 'mape_all_horizons': 44.02460917842708, 'mae_days': 2.718024821688042}
{'dio_change': 18.111473083496094, 'dso_change': 9.910873413085938, 'dpo_change': -0.824144721031189, 'ccc_change': 28.84649121761322}
{'predicted_ccc_days': 100.8, 'covenant_threshold_days': 90, 'breach_predicted': True, 'margin_days': -10.8}


## Supply-chain-enhanced credit risk scorer / SC-PD (Deliverable D2.3.3, Section A8.2)

In [4]:
from src.financial.credit_risk_scorer import train_credit_risk_scorer, explain_prediction
credit_result = train_credit_risk_scorer(nodes)
print(credit_result['metrics'])

sample_id = nodes[nodes.node_type == 'supplier'].node_id.iloc[0]
explanation = explain_prediction(credit_result['model_sc_enhanced'], credit_result['feats'], sample_id)
explanation

  [Credit Risk Scorer] AUC financial-only=0.5447  AUC SC-enhanced=0.8699  (+0.3252)
{'auc_financial_only': 0.5447154471544715, 'auc_sc_enhanced_xgb': 0.8699186991869918, 'auc_improvement': 0.3252032520325203, 'auc_sc_pd_formula': 0.5691056910569106, 'n_sc_features': 16, 'statistically_meaningful_improvement': True}


{'node_id': 'SUPP_0000',
 'predicted_pd': 0.026102690026164055,
 'base_value': 1.0874013900756836,
 'top_contributions': {'inventory_turnover': -2.2136754989624023,
  'supplier_concentration_hhi': -1.1898490190505981,
  'ccc_days': -0.9921704530715942,
  'customer_concentration_hhi': -0.7340514659881592,
  'clustering_coefficient': 0.6314103007316589,
  'ebitda_margin': -0.48140063881874084,
  'interest_coverage': 0.33715593814849854,
  'lead_time_std': -0.2523373067378998}}